# Halftone Simulator

This notebook builds a synthetic image, converts it to CMYK, and explores classic print-style halftone screening. It compares AM and FM halftones, demonstrates dot gain, screen angles, and finally combines the screened CMYK channels into a simple printed-look composite.

In [ ]:
import sys
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

sys.path.insert(0, '../tools')

try:
    import halftone_renderer as hr
    RGBtoCMYK = hr.RGBtoCMYK
    dot_gain = hr.dot_gain
    halftone_screen_am = hr.halftone_screen_am
    halftone_screen_fm = hr.halftone_screen_fm
    print('Imported halftone_renderer from ../tools')
except Exception as exc:
    print(f'Using inline fallback functions: {exc}')

    def RGBtoCMYK(image):
        arr = np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0
        k = 1.0 - arr.max(axis=2)
        denom = np.where(k < 0.999, 1.0 - k, 1.0)
        c = np.where(k < 0.999, (1.0 - arr[..., 0] - k) / denom, 0.0)
        m = np.where(k < 0.999, (1.0 - arr[..., 1] - k) / denom, 0.0)
        y = np.where(k < 0.999, (1.0 - arr[..., 2] - k) / denom, 0.0)
        return tuple(np.clip(ch, 0.0, 1.0) for ch in (c, m, y, k))

    def dot_gain(channel, gain=0.15):
        return np.clip(channel + gain * channel * (1.0 - channel) * 2.0, 0.0, 1.0)

    def _screen_threshold(shape, fx, fy):
        dx = np.abs(fx - 0.5)
        dy = np.abs(fy - 0.5)
        if shape == 'square':
            return np.clip(np.maximum(dx, dy) / 0.5, 0.0, 1.0)
        if shape == 'elliptical':
            return np.clip(np.sqrt((dx / 0.52) ** 2 + (dy / 0.34) ** 2), 0.0, 1.0)
        return np.clip(np.sqrt(dx ** 2 + dy ** 2) / np.sqrt(0.5 ** 2 + 0.5 ** 2), 0.0, 1.0)

    def halftone_screen_am(channel, lpi=85, angle=0, dot_shape='round', dpi=300):
        h, w = channel.shape
        yy, xx = np.indices((h, w))
        theta = np.deg2rad(angle)
        xr = xx * np.cos(theta) + yy * np.sin(theta)
        yr = -xx * np.sin(theta) + yy * np.cos(theta)
        cell = max(4, int(round(dpi / max(lpi, 1))))
        fx = (xr % cell) / cell
        fy = (yr % cell) / cell
        threshold = _screen_threshold(dot_shape, fx, fy)
        return (channel >= threshold).astype(np.float32)

    def _ign_noise(h, w):
        yy, xx = np.indices((h, w))
        return np.mod(52.9829189 * np.mod(0.06711056 * xx + 0.00583715 * yy, 1.0), 1.0)

    def halftone_screen_fm(channel, seed=0):
        h, w = channel.shape
        rng = np.random.default_rng(seed)
        noise = (_ign_noise(h, w) + rng.random((h, w)) * 0.15) % 1.0
        return (channel >= noise).astype(np.float32)

plt.rcParams['figure.figsize'] = (8, 8)
plt.rcParams['image.cmap'] = 'gray'

## Creating a Test Image

A synthetic test target is useful because it contains controlled gradients and recognizable shapes. The vertical gradient stresses mid-tone behavior, while circles and rectangles make it easy to see edge handling, dot growth, and screen artifacts.

In [ ]:
width = height = 512
gradient = np.tile(np.linspace(0, 255, height, dtype=np.uint8)[:, None], (1, width))
rgb = np.dstack([gradient, np.flipud(gradient), gradient])
test_image = Image.fromarray(rgb, mode='RGB')

draw = ImageDraw.Draw(test_image, 'RGBA')
draw.ellipse((60, 60, 220, 220), fill=(255, 80, 80, 180))
draw.ellipse((290, 80, 470, 260), fill=(80, 200, 255, 170))
draw.ellipse((140, 280, 360, 500), fill=(255, 220, 80, 160))
draw.rectangle((40, 380, 160, 500), fill=(40, 40, 40, 180))

plt.figure(figsize=(6, 6))
plt.imshow(test_image)
plt.title('Synthetic gradient + shape test image')
plt.axis('off')
plt.show()

## RGB to CMYK Conversion

Printing is subtractive: cyan, magenta, yellow, and black inks absorb light rather than emit it. A simple RGB-to-CMYK conversion estimates how much of each ink is needed for every pixel before screening turns those continuous values into printable dots.

In [ ]:
c, m, y, k = RGBtoCMYK(test_image)
channels = [('Cyan', c), ('Magenta', m), ('Yellow', y), ('Black', k)]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, (name, channel) in zip(axes.ravel(), channels):
    ax.imshow(channel, cmap='gray', vmin=0, vmax=1)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Dot Gain Simulation

Real ink dots spread on paper, especially on absorbent stock. Dot gain makes the printed result darker than the ideal screen, and it is most visible in highlights and mid-tones where dots grow into surrounding paper.

In [ ]:
c_gain = dot_gain(c, gain=0.15)
m_gain = dot_gain(m, gain=0.15)
y_gain = dot_gain(y, gain=0.15)
k_gain = dot_gain(k, gain=0.15)

before_after = [
    ('C before', c), ('M before', m), ('Y before', y), ('K before', k),
    ('C after', c_gain), ('M after', m_gain), ('Y after', y_gain), ('K after', k_gain),
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (title, channel) in zip(axes.ravel(), before_after):
    ax.imshow(channel, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## AM Halftone at Different LPI

In amplitude-modulated (AM) screening, dots sit on a regular grid while their size changes with tone. A lower LPI gives larger, more visible dots; a higher LPI gives finer detail but demands more print resolution and tighter registration.

In [ ]:
lpis = [85, 133, 200]
am_results = [halftone_screen_am(k_gain, lpi=lpi, angle=0, dot_shape='round') for lpi in lpis]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, lpi, result in zip(axes, lpis, am_results):
    ax.imshow(result, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'K channel at {lpi} LPI')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Dot Shapes - round, square, elliptical

The threshold geometry defines the resulting dot shape. Round dots are the classic general-purpose choice, square dots emphasize a blockier texture, and elliptical dots can help tonal continuity through the mid-tones.

In [ ]:
shapes = ['round', 'square', 'elliptical']
shape_results = [halftone_screen_am(k_gain, lpi=85, angle=0, dot_shape=shape) for shape in shapes]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, shape, result in zip(axes, shapes, shape_results):
    ax.imshow(result, cmap='gray', vmin=0, vmax=1)
    ax.set_title(shape.title())
    ax.axis('off')
plt.tight_layout()
plt.show()

## FM (Stochastic) Halftone

Frequency-modulated (FM) or stochastic screening keeps dots relatively small but varies their spatial distribution. That reduces regular rosette structure and can preserve detail, though it also creates a more grain-like texture.

In [ ]:
am_k = halftone_screen_am(k_gain, lpi=133, angle=0, dot_shape='round')
fm_k = halftone_screen_fm(k_gain, seed=7)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(am_k, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('AM halftone')
axes[0].axis('off')
axes[1].imshow(fm_k, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('FM halftone')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## CMYK Screen Angles

CMYK printing rotates each screen to reduce moiré and produce the familiar rosette structure. A classic angle set is C=15°, M=45°, Y=75°, and K=0°, balancing visual neutrality and interference between channels.

In [ ]:
screen_c = halftone_screen_am(c_gain, lpi=133, angle=15, dot_shape='round')
screen_m = halftone_screen_am(m_gain, lpi=133, angle=45, dot_shape='round')
screen_y = halftone_screen_am(y_gain, lpi=133, angle=75, dot_shape='round')
screen_k = halftone_screen_am(k_gain, lpi=133, angle=0, dot_shape='round')

screens = [('C 15°', screen_c), ('M 45°', screen_m), ('Y 75°', screen_y), ('K 0°', screen_k)]
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, (title, screen) in zip(axes.ravel(), screens):
    ax.imshow(screen, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Combined CMYK Simulation

To approximate the printed result, we combine the screened coverage of each channel as a subtractive composite. This is not a full color-managed press simulation, but it is a useful visual model for how the individual plates interact.

In [ ]:
simulated_rgb = np.dstack([
    1.0 - np.clip(screen_c + screen_k, 0.0, 1.0),
    1.0 - np.clip(screen_m + screen_k, 0.0, 1.0),
    1.0 - np.clip(screen_y + screen_k, 0.0, 1.0),
])

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(test_image)
axes[0].set_title('Original synthetic image')
axes[0].axis('off')
axes[1].imshow(simulated_rgb)
axes[1].set_title('Combined CMYK halftone simulation')
axes[1].axis('off')
plt.tight_layout()
plt.show()